# 상품 분류 헤드 학습 — 실행 드라이버

`fashion_clip` 백본을 **동결**하고 그 위 선형 헤드만 학습한다.
지시서 `../CLASSIFIER_FINETUNE_BRIEF.md`, 설계 `.claude/plans/ai/CH014-class-finetune.md`.

## ⚠️ 이 노트북은 얇은 드라이버다

무거운 단계는 전부 옆의 `.py` 스크립트가 하고, 여기서는 **호출하고 결과 표를 보는 것만** 한다.
브리핑 §3-1: 원격 Jupyter 커널은 연결이 끊기면 죽는다. 오래 걸리는 단계(scan, 임베딩 추출)는
**`tmux` 안에서 같은 명령을 터미널로 돌리는 걸 권장**한다 — 아래 각 셀이 실행할 명령을
그대로 찍어 주므로 복사해서 쓰면 된다.

In [ ]:
# ── 여기만 서버 환경에 맞게 고친다 ────────────────────────────────────
import json
import subprocess
import sys
from pathlib import Path

# 이 노트북이 있는 폴더를 **절대경로로** 적는다.
# 상대경로나 Path.cwd()를 쓰지 않는 이유(브리핑 §3-1): VS Code에서 원격 커널에 붙으면
# 파일은 로컬·실행은 원격이라 상대경로가 원격 기준으로 해석돼 조용히 어긋난다.
HERE = Path("/path/to/repo/AI/validation/CLASS/product-info-finetune")
assert (HERE / "aihub_dataset.py").exists(), f"HERE가 틀렸다: {HERE}"

CLOTHES_ROOT = Path("/path/to/aihub/008.의류_통합_데이터")
ACCESSORY_ROOT = Path("/path/to/aihub/009.패션_액세서리_착용_데이터")
IMAGE_ROOT = Path("/path/to/aihub")
ARTIFACTS = HERE / "artifacts"

VARIANT = "masked"
SHORT_PANTS_MAX_CM = None  # scan 히스토그램을 보고 정할 것. None이면 SHORT_PANTS 파생 안 함

ARTIFACTS.mkdir(parents=True, exist_ok=True)


def run(script, *args):
    """스크립트를 실행하고 출력을 그대로 보여준다. 실행할 명령도 같이 찍는다."""
    cmd = [sys.executable, str(HERE / script), *[str(a) for a in args]]
    print("$ " + " ".join(cmd) + "\n")
    result = subprocess.run(cmd, capture_output=True, text=True, encoding="utf-8")
    print(result.stdout)
    if result.returncode != 0:
        print("--- stderr ---")
        print(result.stderr)
    return result

## STEP 1 — scan (착수 게이트 2개)

이미지 없이 라벨 JSON만 훑는다. 확인할 것:
1. **게이트 1** `model.type`의 front/back이 상품컷인가 착용샷인가 → 찍힌 경로를 직접 열어 볼 것
2. **게이트 2** `clothes.color`에 Navy가 실재하는가

덤으로 `pants_length` 히스토그램(SHORT_PANTS 임계), polo 건수, 그룹 크기 분포가 나온다.
550k라 오래 걸린다 — 감만 볼 거면 `--limit 5000`을 붙일 것.

In [ ]:
run(
    "aihub_dataset.py",
    "scan",
    "--root",
    CLOTHES_ROOT,
    "--accessory-root",
    ACCESSORY_ROOT,
    "--out",
    ARTIFACTS / "scan.txt",
)

## STEP 2 — index (라벨 인덱스 + 분할 + 층화 샘플)

여기서 **어떤 이미지를 받을지**가 정해진다. 소스별 비중은 `--category-cap`(의류)과
`--accessory-cap`(액세서리)으로 조절한다. 출력 맨 아래 "소스별 비중" 표를 보고 다시 돌리면 된다.

카테고리 상한은 **엄수**된다(색상 쿼터가 상한을 넘기지 못한다). 옛 동작이 필요하면
`--allow-cap-overflow`.

In [ ]:
index_args = [
    "index",
    "--root",
    CLOTHES_ROOT,
    "--accessory-root",
    ACCESSORY_ROOT,
    "--out-dir",
    ARTIFACTS,
    "--category-cap",
    3000,
    "--accessory-cap",
    3000,
    "--color-cap",
    2000,
    "--eval-cap",
    100,
]
if SHORT_PANTS_MAX_CM is not None:
    index_args += ["--short-pants-max-cm", SHORT_PANTS_MAX_CM]

run("aihub_dataset.py", *index_args)

### 마스크 크롭이 제대로 되는지 눈으로 확인 (선택)

`__raw.jpg` / `__masked.jpg` 쌍을 몇 장 떨군다. 배경이 실제로 흰색으로 치환됐는지 본다.

In [ ]:
run(
    "aihub_preprocess.py",
    "--index",
    ARTIFACTS / "sample.jsonl",
    "--label-root",
    CLOTHES_ROOT,
    "--image-root",
    IMAGE_ROOT,
    "--out-dir",
    ARTIFACTS / "crop_preview",
    "--count",
    8,
)

## STEP 3 — 임베딩 추출 (GPU, 파이프라인에서 유일하게 무거운 단계)

**이 셀은 tmux에서 돌리는 걸 강력히 권장한다.** 5만 장이면 GPU에서도 수십 분이다.
출력된 `$ ...` 명령을 복사해 터미널에서 실행해도 결과는 같다.

브리핑 §3(A)가 요구하는 **입력 변형 2개 비교**를 위해 `masked`와 `raw`를 둘 다 뽑는다.

In [ ]:
for split in ("train", "val", "test"):
    for variant in ("masked", "raw"):
        run(
            "aihub_embed.py",
            "--index",
            ARTIFACTS / "sample.jsonl",
            "--label-root",
            CLOTHES_ROOT,
            "--image-root",
            IMAGE_ROOT,
            "--out-dir",
            ARTIFACTS,
            "--split",
            split,
            "--variant",
            variant,
        )

## STEP 4 — 헤드 학습 (CPU, 수 초)

- `color` : 14-way 로지스틱 회귀 (커버리지 100%라 가장 확실하다)
- `polo`  : TSHIRT vs SHIRT 이진. 35-way를 건드리지 않아 회귀 위험이 없다
- `category` : 커버 클래스 N-way (미커버 클래스는 계속 zero-shot)

`--class-weight balanced`가 층화 상한과 함께 쓰는 **두 번째 비중 손잡이**다.

In [ ]:
for task in ("color", "polo", "category"):
    run(
        "aihub_train_head.py",
        "--artifacts",
        ARTIFACTS,
        "--task",
        task,
        "--variant",
        VARIANT,
        "--class-weight",
        "balanced",
    )

### masked vs raw — 배경 제거가 실제로 값을 하는가

브리핑 §3(A)의 핵심 질문. 두 변형으로 색상 헤드를 각각 학습해 정확도를 비교한다.

In [ ]:
for variant in ("masked", "raw"):
    run(
        "aihub_train_head.py",
        "--artifacts",
        ARTIFACTS,
        "--task",
        "color",
        "--variant",
        variant,
    )

## STEP 5 — 평가 (AI-Hub 홀드아웃 + 우리 정답셋)

둘을 항상 같이 본다. **어긋나면 우리 정답셋을 믿는다**(도메인이 다르다, 브리핑 §11).
정답셋 이미지는 gitignore라 서버에 따로 올려야 한다 — 없으면 그 섹션만 건너뛴다.

In [ ]:
run(
    "aihub_evaluate.py",
    "--artifacts",
    ARTIFACTS,
    "--variant",
    VARIANT,
    "--tasks",
    "color",
    "polo",
    "--ground-truth-dir",
    HERE.parent / "product-info-check",
    "--out",
    ARTIFACTS / "eval_report.md",
)

## 산출물 확인

In [ ]:
for path in sorted(ARTIFACTS.glob("*")):
    size = path.stat().st_size if path.is_file() else 0
    print(f"{size / 1e6:10.2f} MB  {path.name}")

meta_files = sorted(ARTIFACTS.glob("*.meta.json"))
for meta_file in meta_files:
    print("\n--- " + meta_file.name + " ---")
    print(json.dumps(json.loads(meta_file.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))